# ScreamingFace · compose Fusions

Build reusable model panels without running them. A Fusion has two parts:

1. an ordered set of model members that answer the same question;
2. one reducer that turns their answers into the Fusion answer.

Construction and `.url4` compilation are entirely network-free. This notebook needs no Docker,
provider credentials, or benchmark data.

## 1 · Start with model IDs

In [ ]:
import screamingface as sf

SHARED_PROMPT = "Answer the question directly and support the conclusion."

frontier_fusion = sf.Fusion(
    "frontier-trio",
    models=[
        "codex/gpt-5.5",
        "gemini/3.5-flash",
        "claude/sonnet-4.6",
    ],
    prompt=SHARED_PROMPT,
    reducer=sf.reducers.MajorityVote(),
)

frontier_fusion

Use model-ID strings by default. `prompt=` supplies the default prompt to every
member, while the list preserves stable member order.

`MajorityVote()` selects the most common exact answer. Ties resolve by stable member order, and the
reducer makes no additional model call.

## 2 · Inspect the public authoring values

In [ ]:
{
    "models": frontier_fusion.models,
    "model_ids": frontier_fusion.model_ids,
    "reducer": frontier_fusion.reducer,
    "url4": frontier_fusion.url4,
}

These four values are enough to inspect or share the definition:

- `models` preserves the concise strings and any explicit member mappings;
- `model_ids` provides the ordered route IDs alone;
- `reducer` is the immutable reduction strategy; and
- `url4` is the canonical recipe with its `$question` input still unbound.

Reading any of them remains network-free.

## 3 · Override only the members that differ

In [ ]:
specialist_fusion = sf.Fusion(
    "specialist-pair",
    models=[
        "codex/gpt-5.5",
        {
            "model": "gemini/3.5-flash",
            "prompt": "Check the scientific reasoning and answer directly.",
            "params": {"temperature": 0.2, "max_tokens": 512},
        },
    ],
    prompt="Answer the question carefully.",
    reducer=sf.reducers.MajorityVote(),
)

specialist_fusion.models

A member mapping has one required field, `model`, and two optional overrides,
`prompt` and `params`. Members without an explicit prompt inherit the Fusion prompt.

Each parameter value must be a string, integer, finite float, or boolean. `tools` is reserved: tool
requirements belong to the benchmark so ScreamingFace can apply them consistently to every
answer-producing member.

## 4 · The same model can be more than one member

In [ ]:
SELF_MODEL = "claude/sonnet-4.6"

self_fusion = sf.Fusion(
    "claude-independent-samples",
    models=[
        {
            "model": SELF_MODEL,
            "prompt": "Solve independently and favor precise evidence.",
            "params": {"temperature": 0.2},
        },
        {
            "model": SELF_MODEL,
            "prompt": "Challenge the obvious answer and check alternatives.",
            "params": {"temperature": 0.8},
        },
    ],
    reducer=sf.reducers.Model(
        model="codex/gpt-5.5",
        prompt="Synthesize the strongest supported answer from the panel.",
        params={"temperature": 0.0, "max_tokens": 512},
    ),
)

self_fusion.model_ids

Members are positions in the panel, not unique model names. Repeating a route is
therefore valid: the two Claude members become separate ordered requests with separate prompts and
parameters.

`Model(...)` makes one additional model call. It receives the original question plus every labeled
member answer and synthesizes the Fusion answer. Its `model`, `prompt`, and `params` use the same
model-call vocabulary as an explicit member mapping, but the reducer remains an explicit typed
strategy rather than another panel member.

## 5 · Inspect the self-Fusion recipe

In [ ]:
{
    "models": self_fusion.models,
    "model_ids": self_fusion.model_ids,
    "reducer": self_fusion.reducer,
    "url4": self_fusion.url4,
}

The recipe records the ordered member calls and the reducer call, but it still does
nothing until a concrete question is supplied through execution.

## What construction does not prove

Fusion construction validates the local value shape only. It cannot establish that a configured
engine advertises these routes, supports a benchmark's required tools, or has working provider
credentials. That compatibility is checked when execution begins.

## Recap

- prefer model-ID strings for ordinary members;
- use a mapping only for a member-specific prompt or parameters;
- member order is stable and model IDs may repeat;
- `MajorityVote()` is deterministic and adds no model call;
- `Model(...)` makes one additional synthesis call; and
- construction plus `.url4` inspection are network-free.

Continue to the quickstart to evaluate a Fusion or to the custom-benchmark guide to define your own
cases and scoring contract.